#   `lesson5`:  Text Statistics & Generation


:  sentiment analysis, for instance, or trend discovery.

We will start off by analyzing the relative frequency of words in William Shakespeare's corpus.  We will compare this to [Zipf's law](https://en.wikipedia.org/wiki/Zipf%27s_law), which states that, "the frequency of any word is inversely proportional to its rank in the frequency table."  In other words, the most frequent word is about twice as common as the next frequent word and three times as common as the third most frequent word.

For instance, the word frequency plot of the data on Wikipedia is:

![](https://upload.wikimedia.org/wikipedia/commons/b/b9/Wikipedia-n-zipf.png)

A perfect Zipf's law fit would be a straight line with some constant in the denominator's exponent, but that's not a bad fit at all.

"Formally, let:
* $N$ be the number of elements;
* $k$ be their rank;
* $s$ be the value of the exponent characterizing the distribution.

"Zipf's law then predicts that out of a population of $N$ elements, the normalized frequency of elements of rank $k$, $f(k;s,N)$, is:


$$
f(k;s,N)
=
\frac{1/k^s}{\sum\limits_{n=1}^N (1/n^s)}
$$

In [ ]:
shakespeare_url = 'https://www.gutenberg.org/files/100/100-0.txt'

import requests
shakespeare_data = requests.get( shakespeare_url )

shakespeare_text = shakespeare_data.text

[Project Gutenberg](https://www.gutenberg.org/) texts frequently have a header and footer delimited from the rest of the text by three asterisks `***` before and a number of blank lines afterwards.  (It's not 100% consistent across texts.)  We will clean the text first by stripping those off.

In [ ]:
header_marker = '*** START OF THIS PROJECT GUTENBERG EBOOK THE COMPLETE WORKS OF WILLIAM SHAKESPEARE ***'
header_end = shakespeare_text.find( header_marker ) + len( header_marker )

footer_marker = 'FINIS'
footer_start = shakespeare_text.rfind( footer_marker ) + len( footer_marker )

text = shakespeare_text[ header_end:footer_start ]
print( text[ :90 ] )
print( '---' )
print( text[ -90: ] )

We need to decide what to do with act/scene annotations, stage directions, speaking indicators, numbers, and line breaks (`'\r'` v. `'\n'`).

-   Act/scene annotations have the form `ACT_4|SC_14`.  We can match these using _regular expressions_, which is a way of matching specific text structures.  In this case, we will match and replace entries of the form, `'ACT_{0..9}|SC_{0..9}{0..9}'`.  There aren't many of these, but we'll clean them up.

-   Stage directions (_Exeunt_) are difficult to detect.  We'll leave them in.

-   Speaking indicators (CLEOPATRA.) are also very hard to consistently identify in arbitrary text.  Sometimes they are on a line by themselves, but sometimes they are not.  We'll leave them in.

-   Numbers only occur in acts, scenes, and pages.  We will remove them.

In [ ]:
# Remove act/scene annotations.
import re

text_no_act_nums = re.sub( r'ACT_[0-9]','',text )
text_no_acts     = text_no_act_nums.replace( 'ACT_','' )
text_no_sc_nums  = re.sub( r'SC_[0-9+][0-9]','',text_no_acts )
text_no_scenes   = text_no_sc_nums.replace( 'SC_','' )

# Remove numbers.
text_clean = re.sub( r'[0-9]','',text_no_scenes )

In [ ]:
text_clean

Now we can break the text into pieces by whitespace (easy) and by punctuation (tedious).

In [ ]:
# Identify all non-alphabet, non-whitespace, non-digit characters, the punctuation.
punctuation = set()
for i in text_clean:
    if not i.isalpha() and i not in ( '\r','\n','\t',' ' ):
        punctuation.add( i )
punctuation.remove( '\'' )
punctuation.remove( '-' )
print( punctuation )

In [ ]:
for mark in punctuation:
    text_clean = text_clean.replace( mark,'' )
words = text_clean.lower().split()

In [ ]:
words

Now we have the list of words.  Turning this into a dictionary by count should be old hat for you.

-   Make a dictionary which contains each word as a key with the corresponding number of times the word occurs in `text_clean`.

We turn this into a Pandas DataFrame sorted by frequency.  This makes plotting the text data easier.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline
mpl.rcParams[ 'figure.figsize' ] = (10,10)

sorted_by_value = sorted( word_count.items(),key=lambda kv: kv[ 1 ] )[ ::-1 ]
df = pd.DataFrame( sorted_by_value,columns=('Word','Frequency') )

In [ ]:
df

In [ ]:
df['Frequency'].plot( linestyle='',marker='.',loglog=True )

We can fit Zipf's law to this either as an overall match or piecewise where the distribution curves down in log–log space.  Recall our definition.

$$
f(k;s,N)
=
\frac{1/k^s}{\sum\limits_{n=1}^N (1/n^s)}
$$

In [ ]:
import numpy as np

def f( k,s,N ):
    numer = k ** -s
    denom = np.sum( [ n ** -s for n in range( 1,N+1 ) ] )
    return numer/denom

Let's just pick $s = 1$ for now.  We'll fit it later, but let's get the plot working first.

In [ ]:
N = len( words )
s = 1.0
k = np.logspace( 0,5 )

In [ ]:
plt.plot( k,f(k,s,N ))
df[ 'Frequency' ].plot( linestyle='',marker='.' )

The linear–linear plot is hard to read.  Switch to a log–log plot:

In [ ]:
plt.plot( k,f( k,1.0,N ) * ( 1e6 ) )  # scale up so the line coincides
df[ 'Frequency' ].plot( linestyle='',marker='.',loglog=True )

An actual regression to obtain the coefficient $s$ can be carried out.  There are many packages in Python libraries which can perform a fit.  We will use [`scipy.optimize.curve_fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html).

In [ ]:
from scipy.optimize import curve_fit

x = np.log( np.array( df.index ) + 2 )  # only rank matters, so throw away -inf and 0
y = np.log( df[ 'Frequency' ] )

def f_fit( k,s ):
    numer = k ** -s
    N = len( words )
    denom = np.sum( [ n ** -s for n in range( 1,N+1 ) ] )
    return numer/denom

popt,pcov = curve_fit( f_fit,x,y )

In [ ]:
popt

In [ ]:
plt.plot( k,f( k,popt[ 0 ],N ) * ( 1e6 )  )  # scale up so the line coincides
df[ 'Frequency' ].plot( linestyle='',marker='.',loglog=True )

That's a respectable fit, I suppose, but if we break the regime into two parts (as with the Wikipedia example) then we can do much better.  We will arbitrarily select $10^2$ as the dividing line.

-   Calculate the two values of $s$ for subsets from $[0,100]$ and $[101,N_{\text{words}}]$.  (You will probably need to define two new `f_fit` equivalents with the proper $N$.)

-   Plot the resulting two lines and the word frequency plot as above.

##  Natural Language Processing

We can carry out other interesting text classification and statistical tasks.  One tool that is helpful in this regard is the [Natural Language ToolKit](http://www.nltk.org/), which implements many natural language processing tasks like identifying parts of speech.

In [ ]:
!pip install nltk

import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

We will use these on elements of the Shakespearean corpus.  First, we select a single scene to limit the scope.

In [ ]:
scene = text_clean[ 180008:182199 ]
print( scene )

NLTK can tokenize the words (similar to how we broke things up above).  It can then tag the tokens according to inferred part of speech.

In [ ]:
tokens = nltk.word_tokenize( scene )
print( tokens )

In [ ]:
tagged = nltk.pos_tag( tokens )
print( tagged )

Parts of speech are identified using adjacent words, and the predictive model is quite good for an automatic process.  Consider that it can distinguish `act` as a verb from `act` as a noun, for instance.  It trips up on the archaic command (verb) `Exeunt`, though, identifying it as a proper noun.

The tags of parts of speech [can be very complicated](http://www.nltk.org/book/ch05.html).  Look for things like prepositions `IN`, verbs `VBP`, and nouns `NN`.

With contemporary languages, NLTK can perform reasonably well at identifying key entities (people, organizations, etc.).  Shakespearean English throws it for a bit of a loop:  notice below how `HELENA` becomes an organization (presumably due to all-caps).

In [ ]:
entities = nltk.chunk.ne_chunk(tagged)

print( entities )

##  Markov-Chain Text Generation

Given a sufficiently large corpus of words, we can train a Markov chain generator (a kind of machine learning algorithm) to randomly generate similar text.

A Markov chain represents a series of states with transition probabilities between them.  For instance, for a hypothetical text, the following possible sentence fragments can be generated:

![](./img/markov-chain.png)

That is, at each point in the graph, a random value is generated and used to select one of several possible following states.  The example above is simply a tiny subset of the possible phrases (which in practice would be complicated by our omission of punctuation).

We can train a Markov chain generator by feeding it many possible phrases.  It uses these to assign probabilities to each state and therefore generate "similar" text samples.

_This code is based on [Eli Bendersky's Markov text generator](https://eli.thegreenplace.net/2018/elegant-python-code-for-a-markov-chain-text-generator/)._

In [ ]:
from collections import defaultdict, Counter
import random
import sys

# This is the length of the "state" the current character is predicted from.
# For Markov chains with memory, this is the "order" of the chain. For n-grams,
# n is STATE_LEN+1 since it includes the predicted character as well.
STATE_LEN = 4

data = text_clean.lower()
model = defaultdict( Counter )

print( 'Learning model...' )
for i in range( len( data )-STATE_LEN ):
    state = data[ i:i+STATE_LEN ]
    next  = data[   i+STATE_LEN ]
    model[ state ][ next ] += 1

In [ ]:
N_CHARS = 500

print( 'Sampling...' )
state = random.choice( list( model ) )
out = list( state )
for i in range( N_CHARS ):
    out.extend(random.choices( list( model[ state ] ),model[ state ].values() ) )
    state = state[ 1: ] + out[ -1 ]
print( ''.join( out ) )

This version is based on the _letters_ only, which means that it does some strange things, like create new words.  I saw `"conventur'd"` and `"leavenge"`, for instance.

A better version would use words instead of letters.  Let's write one.

In [ ]:
from collections import defaultdict, Counter
import random
import sys

# This is the length of the "state" the current character is predicted from.
# For Markov chains with memory, this is the "order" of the chain. For n-grams,
# n is STATE_LEN+1 since it includes the predicted character as well.
STATE_LEN = 4

data = words
model = {}

print( 'Learning model...' )
for i in range( len( data )-STATE_LEN ):
    state = data[ i:i+STATE_LEN ]
    next  = data[   i+STATE_LEN ]
    if tuple( state ) in model:
        if next in model[ tuple( state ) ]:
            model[ tuple( state ) ][ next ] += 1
        else:
            model[ tuple( state ) ][ next ] = 1
    else:
        model[ tuple( state ) ] = {}
        model[ tuple( state ) ][ next ] = 1

In [ ]:
N_WORDS = 50

print( 'Sampling...' )
state = random.choice( list( model ) )
out = list( state )
for i in range( N_CHARS ):
    out.extend( random.choices( list( model[ state ] ),model[ state ].values() ) )
    state = state[ 1: ] + ( out[ -1 ], )
print( ' '.join( out ) )

If you spot-check this, you'll find that there are several 5-word or 6-word direct quotes from Shakespeare.  This occurs when there is only one high-likelihood chain to follow.

In [ ]:
model

In [ ]:
sorted_model = sorted( model )

In [ ]:
offset = 735000
sorted_model[ offset:offset+100 ]

-   Now, your final task is to compose a Markov chain generator which trains on _titles_ of Shakespeare plays and predicts new possible titles.  (This will be a fairly restricted space of possible titles.)  You may train this on letters or on words; letters will be more interesting, words will be more accurate.  You will need to extract the correct subset from the Shakespeare corpus above and clean it properly.  You should be able to use one of the previous Markov chain generators with little modification though.